# Assignment 3 - Exploratory field analysis (EDA)

## Purpose

This notebook performs an executable exploratory analysis of the real Story
County, Iowa field sample produced in Assignment 2 and commits three
reproducible visualizations to `docs/assets/`. Every observation below is
computed from the committed Assignment 2 products; no synthetic data is used.

## Sources

- Field polygons: USDA ACPF **Iowa Field Boundaries 2019**
  (`data/processed/assignment-02/fields_with_crops.geojson`), 25 fields selected
  one per 5 x 5 grid cell over Story County (FIPS 19169).
- Crop history: USDA NASS **Cropland Data Layer (CDL)** 2020-2023, Story County
  (`data/processed/assignment-02/cdl_EPSG4326.csv`), one majority-crop summary
  per field per year.

## ACPF temporal limitation

The field boundary layer is a **2019 snapshot** derived from historical FSA
Common Land Unit data. The CDL crop observations cover 2020-2023, which
postdates that snapshot, so any field splits, merges, or boundary edits after
2019 are not represented in this analysis. Field areas and crop summaries
therefore describe 2019 boundaries overlaid with later crop years, and these
polygons do not represent current ownership or program boundaries.


## Load Assignment 2 products and assert 25 unique fields


In [1]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

# nbconvert starts the kernel in the notebook directory; locate the repo root
# by walking upward to the committed Assignment 2 products.
ROOT = Path.cwd()
for parent in (ROOT, *ROOT.parents):
    if (parent / "data" / "processed" / "assignment-02").is_dir():
        ROOT = parent
        break

ASSETS = ROOT / "docs" / "assets"
ASSETS.mkdir(parents=True, exist_ok=True)

FIELDS_PATH = ROOT / "data/processed/assignment-02/fields_with_crops.geojson"
CROPS_PATH = ROOT / "data/processed/assignment-02/cdl_EPSG4326.csv"

fields = gpd.read_file(FIELDS_PATH)
crops = pd.read_csv(CROPS_PATH)

assert fields["field_id"].nunique() == 25, "expected 25 unique fields"
assert len(fields) == 25, "expected 25 field features"
assert crops["field_id"].nunique() == 25, "expected 25 fields in crop table"
assert len(crops) == 25 * 4, "expected one crop record per field per year"

print(f"fields: {len(fields)} features, {fields['field_id'].nunique()} unique field ids")
print(f"crop records: {len(crops)} = {fields['field_id'].nunique()} fields x 4 years")


fields: 25 features, 25 unique field ids
crop records: 100 = 25 fields x 4 years


## Missingness and descriptive statistics


In [2]:
missing = crops[["field_id", "year", "cdl_code", "cdl_name"]].isna().sum()
print("missing values per crop-table column:")
print(missing.to_string())

area_stats = fields["area_ha"].describe()
print("\nfield area (area_ha) descriptive statistics:")
print(area_stats.to_string())

total_area = fields["area_ha"].sum()
median_area = fields["area_ha"].median()
print(f"\ntotal_area_ha = {total_area:.3f}")
print(f"median_area_ha = {median_area:.3f}")


missing values per crop-table column:
field_id    0
year        0
cdl_code    0
cdl_name    0

field area (area_ha) descriptive statistics:
count    25.000000
mean     31.514956
std      17.161441
min       7.618193
25%      17.579540
50%      29.884825
75%      35.916032
max      75.492542

total_area_ha = 787.874
median_area_ha = 29.885


## Field-area histogram


In [3]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

AREA_COLOR = "#4C72B0"

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(fields["area_ha"], bins=8, color=AREA_COLOR, edgecolor="white")
ax.set_title("Distribution of field areas in the 25-field Story County sample")
ax.set_xlabel("Field area (ha)")
ax.set_ylabel("Number of fields")
fig.tight_layout()
fig.text(
    0.01, 0.01,
    "Source: USDA ACPF Iowa Field Boundaries 2019 (field polygons); n = 25 fields.",
    fontsize=8,
)
fig.savefig(ASSETS / "field_area_distribution.png", dpi=160)
plt.close(fig)
print("wrote docs/assets/field_area_distribution.png")


wrote docs/assets/field_area_distribution.png


## 2023 crop-count bar chart


In [4]:
CROP_COLORS = {"Corn": "#FFD300", "Soybeans": "#267000"}

mix_2023 = fields["crop_2023_name"].value_counts().sort_index()
print("2023 majority-crop counts:")
print(mix_2023.to_string())

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(
    mix_2023.index,
    mix_2023.values,
    color=[CROP_COLORS[name] for name in mix_2023.index],
)
ax.bar_label(bars, fmt="%d", padding=3)
ax.set_title("2023 crop mix across the 25-field sample")
ax.set_xlabel("Majority crop (CDL)")
ax.set_ylabel("Number of fields")
ax.set_ylim(0, 25)
fig.tight_layout()
fig.text(
    0.01, 0.01,
    "Source: USDA NASS Cropland Data Layer 2023, Story County "
    "(majority crop per field); n = 25 fields.",
    fontsize=8,
)
fig.savefig(ASSETS / "crop_mix_2023.png", dpi=160)
plt.close(fig)
print("wrote docs/assets/crop_mix_2023.png")


2023 majority-crop counts:
crop_2023_name
Corn        11
Soybeans    14
wrote docs/assets/crop_mix_2023.png


## 2020-2023 crop-sequence frequency chart (rotation patterns)


In [5]:
ROTATION_COLOR = "#55A868"
CODE = {"Corn": "C", "Soybeans": "S", "Oats": "O"}
YEAR_COLUMNS = [
    "crop_2020_name",
    "crop_2021_name",
    "crop_2022_name",
    "crop_2023_name",
]

sequences = fields[YEAR_COLUMNS].apply(
    lambda row: " → ".join(row.map(CODE)), axis=1
)
sequence_counts = sequences.value_counts().sort_values()
print("2020-2023 crop-sequence counts (C = Corn, S = Soybeans, O = Oats):")
print(sequence_counts.to_string())

corn_soy_only = int(
    fields[YEAR_COLUMNS].isin(["Corn", "Soybeans"]).all(axis=1).sum()
)
alternating = int(
    (
        (fields["crop_2020_name"] != fields["crop_2021_name"])
        & (fields["crop_2021_name"] != fields["crop_2022_name"])
        & (fields["crop_2022_name"] != fields["crop_2023_name"])
    ).sum()
)
oats_field = fields.loc[fields["crop_2021_name"] == "Oats"].iloc[0]
print(f"corn/soybeans-only fields: {corn_soy_only} of 25")
print(f"strictly alternating fields: {alternating} of 25")
print(
    f"single non-Corn/Soybeans record: {oats_field['field_id']} "
    f"({oats_field['area_ha']:.1f} ha, Oats in 2021)"
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(sequence_counts.index, sequence_counts.values, color=ROTATION_COLOR)
ax.bar_label(ax.containers[0], fmt="%d", padding=3)
ax.set_title("2020-2023 crop-sequence frequency across the 25-field sample")
ax.set_xlabel("Number of fields")
ax.set_ylabel("Crop sequence (C = Corn, S = Soybeans, O = Oats)")
ax.set_xlim(0, 25)
fig.tight_layout()
fig.text(
    0.01, 0.01,
    "Source: USDA NASS Cropland Data Layer 2020-2023, Story County "
    "(majority crop per field per year); n = 25 fields.",
    fontsize=8,
)
fig.savefig(ASSETS / "crop_rotation_patterns.png", dpi=160)
plt.close(fig)
print("wrote docs/assets/crop_rotation_patterns.png")


2020-2023 crop-sequence counts (C = Corn, S = Soybeans, O = Oats):
C → O → C → S     1
S → C → C → C     1
S → C → C → S     2
C → S → C → C     3
S → C → S → C     7
C → S → C → S    11
corn/soybeans-only fields: 24 of 25
strictly alternating fields: 19 of 25
single non-Corn/Soybeans record: STORY-23 (61.7 ha, Oats in 2021)


wrote docs/assets/crop_rotation_patterns.png


## Observations tied to calculated counts and medians

- The sample has **25 unique fields** and **100 crop records** (one per field
  for each of 2020-2023), with **0 missing crop labels**.
- Field area is skewed: the sample totals **787.9 ha** with a **median of
  29.9 ha**, while areas range from **7.6 ha to 75.5 ha**.
- In 2023, **Soybeans** is the majority crop on **14 fields** and **Corn** on
  **11 fields**; together they account for all 25 fields.
- **24 of 25 fields** planted only Corn or Soybeans across 2020-2023, and
  **19 of 25** alternate between the two every year.
- The modal four-year sequence **Corn -> Soybeans -> Corn -> Soybeans** occurs
  on **11 fields**, followed by the inverse Soybeans -> Corn -> Soybeans ->
  Corn on **7 fields**.
- The single deviation is **STORY-23** (61.7 ha), which grew **Oats in 2021**
  for a Corn -> Oats -> Corn -> Soybeans sequence.
- These figures inherit the **ACPF 2019 boundary limitation** described above:
  crop years 2020-2023 are summarized inside 2019-era polygons.
